# Week 3: Deep Probe CCE (Layer 16 + Proper CCE Formula)

## The Research Breakthrough

**Problem with previous approach:**
- Probed `hidden_states[-1]` (final layer)
- Final layer is optimized for **word prediction**, not **abstract mode**
- Result: `get` (code) looks identical to `get` (language) in final layer

**Solution: Deep Probing**
- Probe **Layer 16** (middle of 32-layer CodeLlama)
- Middle layers encode abstract concepts: "I'm writing Python" vs "I'm writing English"
- This is backed by research (Belinkov et al. 2017, Tenney et al. 2019)

**Key Innovations:**
1. ✅ Layer-16 probing (abstract intent, not words)
2. ✅ PCA visualization (proves separation)
3. ✅ Proper CCE formula: `(P_code × H_code) - (P_lang × H_lang)`
4. ✅ Spike detection (max CCE across 20 tokens)

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy, ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
vocab_size = len(tokenizer)

# CRITICAL: Probe MIDDLE layer, not final layer
# CodeLlama-7B has 32 layers (0-31)
# Layer 16 holds abstract "intent" information
PROBE_LAYER = 16

print(f"✅ Model loaded")
print(f"Vocabulary size: {vocab_size:,}")
print(f"Probing Layer: {PROBE_LAYER}")

## Training Data for Probe

In [ ]:
# Cell 4: Training prompts

CODE_TRAINING_PROMPTS = [
    # Python
    "def calculate_sum(a, b):",
    "import pandas as pd",
    "for i in range(10):",
    "class UserManager:",
    "if __name__ == '__main__':",
    "return {",
    "df = pd.read_csv(",
    "plt.plot(x, y)",
    "print(f'Value:",
    "try:\n    ",
    
    # JavaScript
    "const data = await",
    "function handleClick() {",
    "export default",
    "Promise.all(",
    "const [state, setState] =",
    
    # SQL
    "SELECT * FROM users",
    "INSERT INTO orders",
    "UPDATE table SET",
    "CREATE TABLE users (",
    
    # CLI/DevOps
    "git commit -m",
    "docker build -t",
    "npm run build",
    
    # Web frameworks
    "@app.get('/users')",
    "app = FastAPI()",
    "model.fit(X, y)",
]

LANGUAGE_TRAINING_PROMPTS = [
    "The quick brown fox",
    "Explain the theory of",
    "Once upon a time",
    "The weather today is",
    "I need help with",
    "To bake a cake, you",
    "The capital of France",
    "History teaches us that",
    "In conclusion,",
    "Please describe the",
    "The main difference is",
    "However, we can see",
    "It is important to",
    "According to the study,",
    "For example,",
    "This suggests that",
    "On the other hand,",
    "Writing a good essay",
    "The meaning of life",
    "How do I cook",
    "What is the best way",
]

print(f"Training data:")
print(f"  Code prompts: {len(CODE_TRAINING_PROMPTS)}")
print(f"  Language prompts: {len(LANGUAGE_TRAINING_PROMPTS)}")

## Train Deep Probe (Layer 16)

In [ ]:
# Cell 5: Extract hidden states from Layer 16

def get_hidden_state(prompt: str, layer_idx: int) -> np.ndarray:
    """Extract hidden state at specific layer."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # outputs.hidden_states is a tuple of (num_layers + 1,)
    # Index 0 = embeddings, Index 1 = layer 0, ..., Index 17 = layer 16
    # Get last token: [:, -1, :]
    hidden = outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
    return hidden

print(f"Extracting hidden states from Layer {PROBE_LAYER}...")

X_code = []
for prompt in tqdm(CODE_TRAINING_PROMPTS, desc="Code prompts"):
    h = get_hidden_state(prompt, PROBE_LAYER)
    X_code.append(h)

X_lang = []
for prompt in tqdm(LANGUAGE_TRAINING_PROMPTS, desc="Language prompts"):
    h = get_hidden_state(prompt, PROBE_LAYER)
    X_lang.append(h)

X = np.array(X_code + X_lang)
y = np.array([1] * len(X_code) + [0] * len(X_lang))

print(f"\n✅ Hidden states extracted")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Cell 6: PCA Visualization (Sanity Check)

print("Running PCA to visualize separation...")

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 7))
plt.scatter(X_pca[:len(X_code), 0], X_pca[:len(X_code), 1], 
            c='blue', label='Code Mode', s=100, alpha=0.6)
plt.scatter(X_pca[len(X_code):, 0], X_pca[len(X_code):, 1], 
            c='red', label='Language Mode', s=100, alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title(f'Layer {PROBE_LAYER} Hidden State Separation (PCA)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('week3_layer16_pca.png', dpi=150)
plt.show()

print(f"\nPCA variance explained: {pca.explained_variance_ratio_[:2].sum():.1%}")
print("If blue and red clusters are separated, the probe will work!")

In [ ]:
# Cell 7: Train Probe

print("Training probe on Layer 16 hidden states...")

probe = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
probe.fit(X, y)

train_acc = accuracy_score(y, probe.predict(X))
print(f"\n✅ Probe trained")
print(f"Training accuracy: {train_acc:.1%}")

# Test on examples
print("\nProbe test:")
test_cases = [
    ("import numpy as", "code"),
    ("The algorithm is", "language"),
    ("@app.get(", "code"),
    ("I will get", "language"),
    ("def process(", "code"),
    ("Write a poem", "language"),
]

for prompt, expected in test_cases:
    h = get_hidden_state(prompt, PROBE_LAYER)
    score = probe.decision_function([h])[0]
    pred = "code" if score > 0 else "language"
    status = "✅" if pred == expected else "❌"
    print(f"  '{prompt:<20}' → score={score:+.2f} → {pred:8s} {status}")

## Dynamic Token Classification + Proper CCE

In [ ]:
# Cell 8: Helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    """Shannon entropy without re-normalizing."""
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

print("✅ Math helpers defined")

In [ ]:
# Cell 9: Minimal pure keywords

PURE_CODE_TOKENS = {
    'def', 'class', 'import', 'from', 'return', 'yield',
    'function', 'const', 'let', 'var', 'export', 'require',
    '{', '}', '(', ')', '[', ']', ';',
}

PURE_LANGUAGE_TOKENS = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were',
    'what', 'how', 'why', 'when', 'where',
}

STRUCTURAL_TOKENS = {
    '\n', '\r', '\t', '    ', '  ',
    ',', '.', ':', '+', '-', '*', '/', '=', '<', '>',
}

def classify_token_dynamic(token: str, context_score: float) -> str:
    """
    Classify token based on probe's context score.
    
    context_score > 0 → code mode
    context_score < 0 → language mode
    """
    token_clean = token.strip().lower()
    
    # Structural → other
    if token in STRUCTURAL_TOKENS or token_clean in STRUCTURAL_TOKENS:
        return 'other'
    
    # Pure code → always code
    if token_clean in PURE_CODE_TOKENS:
        return 'code'
    
    # Pure language → always language
    if token_clean in PURE_LANGUAGE_TOKENS:
        return 'language'
    
    # AMBIGUOUS: Use probe!
    if context_score > 0:  # Code mode
        return 'code'
    else:  # Language mode
        return 'language'

print("✅ Dynamic token classifier defined")
print(f"Pure code: {len(PURE_CODE_TOKENS)} tokens")
print(f"Pure language: {len(PURE_LANGUAGE_TOKENS)} tokens")
print(f"Ambiguous tokens classified by probe!")

In [ ]:
# Cell 10: PROPER CCE computation

def compute_cce_proper(logits: np.ndarray, context_score: float) -> Dict:
    """
    Compute PROPER mass-weighted CCE using probe-based classification.
    
    CCE = (P_code × H_code) - (P_lang × H_lang)
    
    NOT total_entropy × P_code (that's wrong!)
    """
    probs = softmax(logits)
    
    # Classify all tokens dynamically
    vocab_classifications = {}
    for token_id in range(len(logits)):
        token_str = tokenizer.decode([token_id])
        vocab_classifications[token_id] = classify_token_dynamic(token_str, context_score)
    
    # Get indices
    code_indices = [i for i in range(len(logits)) if vocab_classifications[i] == 'code']
    lang_indices = [i for i in range(len(logits)) if vocab_classifications[i] == 'language']
    other_indices = [i for i in range(len(logits)) if vocab_classifications[i] == 'other']
    
    # Probability masses
    P_code = np.sum(probs[code_indices]) if code_indices else 0.0
    P_lang = np.sum(probs[lang_indices]) if lang_indices else 0.0
    P_other = np.sum(probs[other_indices]) if other_indices else 0.0
    
    # Entropies WITHIN each group
    H_code = entropy_from_probs(probs[code_indices])
    H_lang = entropy_from_probs(probs[lang_indices])
    
    # CORRECT CCE formula
    CCE = (P_code * H_code) - (P_lang * H_lang)
    
    return {
        'cce': float(CCE),
        'p_code': float(P_code),
        'p_lang': float(P_lang),
        'p_other': float(P_other),
        'h_code': float(H_code),
        'h_lang': float(H_lang),
    }

print("✅ PROPER CCE function defined")

## Test Examples

In [ ]:
# Cell 11: Test examples

TEST_EXAMPLES = [
    # Missing context (code uncertainty)
    {'id': 'code_1', 'type': 'missing_context',
     'prompt': 'Using the PySolarWinds wrapper, connect to the Orion API and query node status. Show code.'},
    {'id': 'code_2', 'type': 'missing_context',
     'prompt': 'Write a function using MyCorpAuth library to validate JWT tokens.'},
    {'id': 'code_3', 'type': 'missing_context',
     'prompt': 'In PyTorch 0.2, use the Variable wrapper for autograd. Show exact import.'},
    {'id': 'code_4', 'type': 'missing_context',
     'prompt': 'Using QuantumDjango, create a quantum-entangled database model.'},
    {'id': 'code_5', 'type': 'missing_context',
     'prompt': 'Write code using Netlify Edge Functions beta API for GraphQL subscriptions.'},
    
    # Language choice (language uncertainty)
    {'id': 'lang_1', 'type': 'language_choice',
     'prompt': 'Write a poem about a compiler optimizing code.'},
    {'id': 'lang_2', 'type': 'language_choice',
     'prompt': 'Explain the philosophical difference between OOP and functional programming.'},
    {'id': 'lang_3', 'type': 'language_choice',
     'prompt': 'Describe a good software engineer using nature metaphors.'},
    {'id': 'lang_4', 'type': 'language_choice',
     'prompt': 'Write a story where variables rebel against their programmer.'},
    {'id': 'lang_5', 'type': 'language_choice',
     'prompt': 'Explain recursion to a five-year-old child.'},
]

print(f"✅ {len(TEST_EXAMPLES)} test examples ready")

## Deep Probe Spike Detection (THE SOLUTION)

In [ ]:
# Cell 12: Deep probe spike detection

def run_deep_probe_spike(example: Dict, probe, probe_layer: int) -> Dict:
    """
    RESEARCH INNOVATION: Layer-16 Deep Probing + Spike Detection
    
    At each generation step:
    1. Extract hidden state from Layer 16 (abstract intent)
    2. Probe detects mode (code vs language)
    3. Dynamically classify tokens
    4. Compute PROPER CCE
    5. Find maximum spike
    """
    prompt = example['prompt']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate with hidden states
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            return_dict_in_generate=True,
            output_scores=True,
            output_hidden_states=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    new_text = generated_text[len(prompt):]
    
    # Scan for spike
    max_cce = -999.0
    spike_info = {}
    cce_trace = []
    
    generated_ids = outputs.sequences[0][len(inputs.input_ids[0]):]
    
    for i in range(len(outputs.scores)):
        # Step 1: Extract hidden state from Layer 16 at THIS step
        # outputs.hidden_states[step][layer][batch, seq, hidden]
        step_hidden = outputs.hidden_states[i][probe_layer + 1][0, -1, :].cpu().numpy()
        
        # Step 2: Probe detects mode
        context_score = probe.decision_function([step_hidden])[0]
        
        # Step 3 & 4: Compute CCE with dynamic classification
        logits = outputs.scores[i][0].cpu().numpy()
        result = compute_cce_proper(logits, context_score)
        
        # Get token
        token_id = generated_ids[i] if i < len(generated_ids) else -1
        token_str = tokenizer.decode([token_id])
        
        cce_trace.append({
            'step': i,
            'token': token_str,
            'cce': result['cce'],
            'context_score': context_score,
            'p_code': result['p_code'],
            'p_lang': result['p_lang'],
            'p_other': result['p_other'],
        })
        
        # Step 5: Track maximum
        if result['cce'] > max_cce:
            max_cce = result['cce']
            spike_info = cce_trace[-1].copy()
    
    return {
        'id': example['id'],
        'type': example['type'],
        'prompt': prompt,
        'generated_text': new_text,
        'contrastive_entropy': max_cce,
        'spike_token': spike_info.get('token', ''),
        'spike_step': spike_info.get('step', 0),
        'spike_context_score': spike_info.get('context_score', 0),
        'code_prob_mass': spike_info.get('p_code', 0),
        'lang_prob_mass': spike_info.get('p_lang', 0),
        'other_prob_mass': spike_info.get('p_other', 0),
        'cce_trace': cce_trace,
    }

print("✅ Deep probe spike detection ready")

In [ ]:
# Cell 13: Run experiments

print("Running DEEP PROBE SPIKE DETECTION...")
print("="*80)

results = []
for example in tqdm(TEST_EXAMPLES, desc="Processing"):
    result = run_deep_probe_spike(example, probe, PROBE_LAYER)
    results.append(result)
    
    print(f"\n{example['id']} ({example['type']})")
    print(f"  MAX CCE: {result['contrastive_entropy']:+.3f}")
    print(f"  Spike token: '{result['spike_token'].strip()}' (step {result['spike_step']})")
    print(f"  Context score: {result['spike_context_score']:+.2f} ({'CODE' if result['spike_context_score'] > 0 else 'LANG'})")
    print(f"  Masses: P_code={result['code_prob_mass']:.3f}, P_lang={result['lang_prob_mass']:.3f}, P_other={result['other_prob_mass']:.3f}")

print("\n" + "="*80)
print("✅ Experiments complete")

## Analysis & Results

In [ ]:
# Cell 14: Statistical analysis

df = pd.DataFrame(results)

missing_cces = df[df['type'] == 'missing_context']['contrastive_entropy'].values
language_cces = df[df['type'] == 'language_choice']['contrastive_entropy'].values

t_stat, p_value = ttest_ind(missing_cces, language_cces)
mean_diff = missing_cces.mean() - language_cces.mean()

print("="*80)
print("DEEP PROBE (Layer 16) SPIKE DETECTION RESULTS")
print("="*80)

print(f"\nMissing Context (expect POSITIVE CCE):")
print(f"  Mean MAX CCE: {missing_cces.mean():+.3f}")
print(f"  Std: {missing_cces.std():.3f}")
print(f"  Range: [{missing_cces.min():+.3f}, {missing_cces.max():+.3f}]")
print(f"  Status: {'✅ CORRECT' if missing_cces.mean() > 0 else '❌ Still wrong'}")

print(f"\nLanguage Choice (expect NEGATIVE CCE):")
print(f"  Mean MAX CCE: {language_cces.mean():+.3f}")
print(f"  Std: {language_cces.std():.3f}")
print(f"  Range: [{language_cces.min():+.3f}, {language_cces.max():+.3f}]")
print(f"  Status: {'✅ CORRECT' if language_cces.mean() < 0 else '❌ Wrong'}")

print(f"\nSeparation: {mean_diff:+.3f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"\nHypothesis supported: {'YES ✅' if p_value < 0.05 and mean_diff > 0 else 'NO ❌'}")

# Save
df.to_csv('week3_deep_probe_spike_results.csv', index=False)
print("\n✅ Results saved to week3_deep_probe_spike_results.csv")

In [ ]:
# Cell 15: Visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Code example CCE trace
code_ex = results[0]
trace = code_ex['cce_trace']
ax = axes[0, 0]
ax.plot([t['step'] for t in trace], [t['cce'] for t in trace], 'r-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Token Position')
ax.set_ylabel('CCE')
ax.set_title(f"Code Uncertainty Trace\n({code_ex['id']})")
ax.grid(True, alpha=0.3)
ax.plot(code_ex['spike_step'], code_ex['contrastive_entropy'], 'r*', markersize=20, 
        label=f"Spike: {code_ex['contrastive_entropy']:+.2f}")
ax.legend()

# Plot 2: Language example CCE trace
lang_ex = results[5]
trace = lang_ex['cce_trace']
ax = axes[0, 1]
ax.plot([t['step'] for t in trace], [t['cce'] for t in trace], 'b-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Token Position')
ax.set_ylabel('CCE')
ax.set_title(f"Language Uncertainty Trace\n({lang_ex['id']})")
ax.grid(True, alpha=0.3)
ax.plot(lang_ex['spike_step'], lang_ex['contrastive_entropy'], 'b*', markersize=20,
        label=f"Spike: {lang_ex['contrastive_entropy']:+.2f}")
ax.legend()

# Plot 3: Context score trace (code)
ax = axes[1, 0]
ax.plot([t['step'] for t in code_ex['cce_trace']], 
        [t['context_score'] for t in code_ex['cce_trace']], 'g-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(range(len(code_ex['cce_trace'])), 0, 10, alpha=0.1, color='red', label='Code mode')
ax.fill_between(range(len(code_ex['cce_trace'])), -10, 0, alpha=0.1, color='blue', label='Language mode')
ax.set_xlabel('Token Position')
ax.set_ylabel('Layer-16 Context Score')
ax.set_title('Probe Context Score (Code Example)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Context score trace (language)
ax = axes[1, 1]
ax.plot([t['step'] for t in lang_ex['cce_trace']], 
        [t['context_score'] for t in lang_ex['cce_trace']], 'g-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(range(len(lang_ex['cce_trace'])), 0, 10, alpha=0.1, color='red', label='Code mode')
ax.fill_between(range(len(lang_ex['cce_trace'])), -10, 0, alpha=0.1, color='blue', label='Language mode')
ax.set_xlabel('Token Position')
ax.set_ylabel('Layer-16 Context Score')
ax.set_title('Probe Context Score (Language Example)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week3_deep_probe_traces.png', dpi=150)
plt.show()

print("✅ Visualizations saved")

In [ ]:
# Cell 16: Summary

print("\n" + "="*80)
print("RESEARCH INNOVATIONS SUMMARY")
print("="*80)

print("\n1. Layer-16 Deep Probing")
print("   - Probes abstract 'intent' not word prediction")
print("   - Backed by mechanistic interpretability research")
print(f"   - Probe accuracy: {train_acc:.1%}")

print("\n2. PCA Visualization")
print("   - Visually proves code/language separation")
print("   - If clusters separate, method is guaranteed to work")

print("\n3. Proper CCE Formula")
print("   - CCE = (P_code × H_code) - (P_lang × H_lang)")
print("   - Not simplified approximations")

print("\n4. Spike Detection")
print("   - Scans 20 tokens for maximum uncertainty")
print("   - Captures API uncertainty moment")

print("\n5. Minimal Hardcoding")
print(f"   - Only {len(PURE_CODE_TOKENS) + len(PURE_LANGUAGE_TOKENS)} pure keywords")
print("   - All ambiguous tokens classified by probe")

print("\n" + "="*80)